# 00 — Understanding Praval Architecture

        **Mode:** `offline`  
        **Version:** Praval `0.8.0`  
        **Course link:** [Watch the original course video](https://www.youtube.com/watch?v=M30U-6w_WGc)

        This notebook makes the framework's execution visible. It records actual
        stages and renders the messages or runtime events that connect them.

        **You will see**

        - A Reef routes structured Spores.
- Subscribers run independently.
- Completion and shutdown are explicit lifecycle stages.

        Run the cells from top to bottom. Committed notebooks contain no saved
        output, so everything shown is produced by your installed Praval package.


In [ ]:
from __future__ import annotations

import html as _html
import json
import os
import time
from pathlib import Path

from IPython.display import HTML, display

os.environ.setdefault("PRAVAL_OBSERVABILITY", "off")


from praval import get_provider_registry
from praval.models import ModelResponse, ProviderCapabilities


class NotebookLifecycleProvider:
    """Credential-free adapter for agents whose handlers do not call a model."""

    provider_name = "notebook-lifecycle"
    capabilities = ProviderCapabilities(text=True)

    def __init__(self, config):
        self.config = config

    def invoke(self, request, tools=None):
        return ModelResponse(
            content="Notebook lifecycle response",
            provider=self.provider_name,
            model=request.model,
        )

    def close(self):
        return None


_notebook_registry = get_provider_registry()
_notebook_registry.register_provider(
    "notebook-lifecycle",
    NotebookLifecycleProvider,
    default_model="notebook-lifecycle-model",
    capabilities=NotebookLifecycleProvider.capabilities,
)
os.environ.setdefault("PRAVAL_DEFAULT_PROVIDER", "notebook-lifecycle")
os.environ.setdefault("PRAVAL_DEFAULT_MODEL", "notebook-lifecycle-model")

EVENTS = []
_STARTED = time.perf_counter()


def stage(actor, action, detail="", spore=None):
    """Capture one observable execution stage."""
    EVENTS.append(
        {
            "ms": round((time.perf_counter() - _STARTED) * 1000, 1),
            "actor": actor,
            "action": action,
            "detail": detail,
            "spore_id": getattr(spore, "id", "") if spore else "",
        }
    )


def show_flow(*steps):
    cards = []
    for index, (name, detail) in enumerate(steps):
        if index:
            cards.append('<div style="font-size:24px;color:#557">→</div>')
        cards.append(
            '<div style="padding:12px 16px;border:1px solid #b8c7dc;'
            'border-radius:12px;background:#f7fbff;min-width:130px">'
            f'<b>{_html.escape(name)}</b><br>'
            f'<span style="color:#456;font-size:12px">'
            f'{_html.escape(detail)}</span></div>'
        )
    display(
        HTML(
            '<div style="display:flex;gap:10px;align-items:center;'
            'flex-wrap:wrap;margin:12px 0">' + "".join(cards) + "</div>"
        )
    )


def show_timeline(events=None):
    events = EVENTS if events is None else events
    rows = []
    for item in events:
        rows.append(
            "<tr>"
            f"<td>{item['ms']:.1f}</td>"
            f"<td><b>{_html.escape(str(item['actor']))}</b></td>"
            f"<td>{_html.escape(str(item['action']))}</td>"
            f"<td>{_html.escape(str(item['detail']))}</td>"
            f"<td><code>{_html.escape(str(item['spore_id'])[:12])}</code></td>"
            "</tr>"
        )
    display(
        HTML(
            '<table style="border-collapse:collapse;width:100%">'
            '<thead><tr><th>ms</th><th>actor</th><th>stage</th>'
            '<th>detail</th><th>spore</th></tr></thead><tbody>'
            + "".join(rows)
            + "</tbody></table>"
        )
    )


def show_spore(spore, label="Spore on the Reef"):
    payload = json.loads(spore.to_json())
    rendered = _html.escape(json.dumps(payload, indent=2, sort_keys=True))
    display(
        HTML(
            '<div style="border-left:5px solid #7b61ff;padding:10px 14px;'
            'background:#faf9ff;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def show_json(value, label="Result"):
    rendered = _html.escape(json.dumps(value, indent=2, sort_keys=True, default=str))
    display(
        HTML(
            '<div style="border-left:5px solid #18a999;padding:10px 14px;'
            'background:#f5fffd;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def require_env(*names):
    missing = [name for name in names if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing required notebook configuration: " + ", ".join(missing))
    return {name: os.environ[name] for name in names}


def find_example_asset(relative):
    relative = Path(relative)
    starts = [Path.cwd(), *Path.cwd().parents]
    for root in starts:
        for candidate in (root / relative, root / "examples" / relative):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"Could not locate example asset: {relative}")


## The three moving parts

An **Agent** specializes in one job. A **Spore** carries structured knowledge.
The **Reef** routes that Spore to interested agents. There is no central agent
deciding every step.


In [ ]:
show_flow(
    ("Producer", "creates knowledge"),
    ("Spore", "structured message"),
    ("Reef", "routes by channel"),
    ("Subscriber", "handles the message"),
)


## Send one real Spore

This uses a fresh in-memory Reef. The observer receives the exact Spore object
stored by the channel.


In [ ]:
from praval.core.reef import Reef

reef = Reef()
received = []


def architecture_observer(spore):
    stage("observer", "handler started", spore.knowledge["type"], spore)
    received.append(spore)
    stage("observer", "handler finished", "knowledge captured", spore)


reef.subscribe(
    "architecture-observer",
    architecture_observer,
    channel=reef.default_channel,
)
stage("learner", "broadcast requested", "architecture_question")
spore_id = reef.broadcast(
    from_agent="learner",
    knowledge={
        "type": "architecture_question",
        "question": "How do agents exchange knowledge?",
    },
)
stage("reef", "Spore accepted", spore_id)
assert reef.wait_for_completion(timeout=10)
assert len(received) == 1


In [ ]:
show_spore(received[0])
show_timeline()
show_json(reef.get_network_stats(), "Reef network statistics")


## What to notice

The handler receives an immutable identity, sender, type, timestamps, and a
JSON-serializable knowledge payload. `wait_for_completion()` accounts for
handler work; `shutdown()` releases the channel executors.


In [ ]:
assert received[0].id == spore_id
assert received[0].from_agent == "learner"
assert received[0].knowledge["question"].startswith("How")
assert reef.shutdown(timeout=10)
stage("reef", "shutdown", "all channel workers closed")
show_timeline()
